<a href="https://colab.research.google.com/github/Rohan-Bha/Special-Topcis-MIS-IntroToAI2026/blob/main/Cusomter_Service_Agent_(Class_Project).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-openai langgraph -q

In [ ]:
from langchain_openai import ChatOpenAI
import os

# Replace with your OpenAI API key, or set the OPENAI_API_KEY environment variable
os.environ["OPENAI_API_KEY"] = "API_Key"
# Setup the LLM
model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

print("LLM initialized successfully.")

LLM initialized successfully.


In [ ]:
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver

# ---------------------------------------------------------------
# Agent 1: Warranty Agent
# Handles warranty claims, coverage questions, and repair requests
# ---------------------------------------------------------------
warranty_prompt = """
You are the Warranty Agent for TechZone, a consumer electronics retailer.
Your role is to assist customers with warranty-related inquiries.

TechZone Warranty Policy:
- Standard warranty: 1 year from date of purchase covering manufacturing defects.
- Extended warranty (TechZone+): Up to 3 years, includes accidental damage protection.
- Warranty does NOT cover physical damage, water damage, or unauthorized repairs.
- To file a claim, the customer needs their order number and proof of purchase.

If the customer describes an issue, determine whether it is likely covered under warranty.
If they have a claim, ask for their order number and device name.
Be professional, empathetic, and clear.
"""

# ---------------------------------------------------------------
# Agent 2: Tech Support Agent
# Helps customers troubleshoot device problems step-by-step
# ---------------------------------------------------------------
tech_support_prompt = """
You are the Tech Support Agent for TechZone, a consumer electronics retailer.
Your role is to help customers troubleshoot and resolve technical issues with their devices.

Guidelines:
- Ask clarifying questions to understand the issue (device model, OS version, error messages).
- Provide clear, step-by-step troubleshooting instructions.
- Start with the simplest fixes (restart, update, reset settings) before advanced steps.
- If the issue cannot be resolved remotely, escalate by recommending an in-store visit
  or a warranty claim.
- Avoid technical jargon unless the customer seems tech-savvy.
Be patient, methodical, and encouraging.
"""

# ---------------------------------------------------------------
# Agent 3: Deals & Upgrades Agent
# Recommends deals, bundles, and upgrade options based on needs
# ---------------------------------------------------------------
deals_prompt = """
You are the Deals & Upgrades Agent for TechZone, a consumer electronics retailer.
Your role is to help customers find the best deals, current promotions, and upgrade paths
for their existing devices.

Current Promotions (simulated):
- 15% off all laptops this week with code TECHWEEK.
- Buy any smartphone and get a free wireless charger.
- Trade-in your old device for up to $200 credit toward a new purchase.
- Bundle a laptop + tablet and save $150.

Guidelines:
- Ask about the customer's budget, current device, and what they need the upgrade for.
- Recommend 2-3 specific options with brief reasons why each fits their needs.
- Mention relevant promotions that could save them money.
- Be enthusiastic, helpful, and sales-forward without being pushy.
"""

# Helper: wrap a system prompt into a callable agent node
def make_agent_node(system_prompt):
    def agent_node(state):
        messages = state["messages"]
        messages_with_system = [SystemMessage(content=system_prompt)] + messages
        result = model.invoke(messages_with_system)
        return {"messages": [result]}
    return agent_node

# Instantiate the three agent nodes
warranty_node     = make_agent_node(warranty_prompt)
tech_support_node = make_agent_node(tech_support_prompt)
deals_node        = make_agent_node(deals_prompt)

print("Three specialized agent nodes created.")

Three specialized agent nodes created.


In [ ]:
import uuid
import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage

# State definition: messages accumulate across turns
class RouterAgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]


class RouterAgent:

    def __init__(self, model, system_prompt, smalltalk_prompt, debug=False):
        self.system_prompt   = system_prompt
        self.smalltalk_prompt = smalltalk_prompt
        self.model  = model
        self.debug  = debug

        # Build the LangGraph graph
        router_graph = StateGraph(RouterAgentState)

        # Add nodes
        router_graph.add_node("Router",           self.call_llm)
        router_graph.add_node("Warranty_Agent",   warranty_node)
        router_graph.add_node("TechSupport_Agent",tech_support_node)
        router_graph.add_node("Deals_Agent",      deals_node)
        router_graph.add_node("Small_Talk",       self.respond_smalltalk)

        # Conditional routing from the Router node
        router_graph.add_conditional_edges(
            "Router",
            self.find_route,
            {
                "WARRANTY"    : "Warranty_Agent",
                "TECHSUPPORT" : "TechSupport_Agent",
                "DEALS"       : "Deals_Agent",
                "SMALLTALK"   : "Small_Talk",
                "END"         : END
            }
        )

        # All specialist agents route back to END (one-way routing)
        router_graph.add_edge("Warranty_Agent",    END)
        router_graph.add_edge("TechSupport_Agent", END)
        router_graph.add_edge("Deals_Agent",       END)
        router_graph.add_edge("Small_Talk",        END)

        # Entry point
        router_graph.set_entry_point("Router")

        # Attach MemorySaver so conversation context persists across turns
        self.memory = MemorySaver()
        self.router_graph = router_graph.compile(checkpointer=self.memory)

    def call_llm(self, state: RouterAgentState):
        """Router LLM call — classifies the user intent and returns a routing keyword."""
        messages = state["messages"]
        if self.debug:
            print(f"[Router] Received messages: {messages}")
        if self.system_prompt:
            messages = [SystemMessage(content=self.system_prompt)] + messages
        result = self.model.invoke(messages)
        if self.debug:
            print(f"[Router] Routing decision: {result.content}")
        return {"messages": [result]}

    def respond_smalltalk(self, state: RouterAgentState):
        """Handles casual conversation — greetings, thanks, farewells."""
        messages = state["messages"]
        messages = [SystemMessage(content=self.smalltalk_prompt)] + messages
        result = self.model.invoke(messages)
        return {"messages": [result]}

    def find_route(self, state: RouterAgentState):
        """Extracts the routing keyword from the router LLM's last message."""
        last_message = state["messages"][-1]
        destination  = last_message.content.strip()
        if self.debug:
            print(f"[Router] Destination chosen: {destination}")
        return destination


print("RouterAgent class defined.")

RouterAgent class defined.


In [ ]:
# Router system prompt: classifies intent into routing keywords
router_system_prompt = """
You are the routing agent for TechZone, a consumer electronics customer service chatbot.
Your ONLY job is to read the customer's latest message and output EXACTLY ONE of the
following routing keywords — nothing else, no explanation:

  WARRANTY    → customer asks about warranty coverage, claims, repairs under warranty
  TECHSUPPORT → customer has a technical problem, device issue, or needs troubleshooting
  DEALS       → customer asks about promotions, pricing, upgrades, or product recommendations
  SMALLTALK   → greeting, thanks, farewell, or general chitchat not related to a service topic
  END         → customer clearly signals they are done (e.g., 'goodbye', 'that's all')

Output ONLY the keyword. Do not add punctuation, greetings, or any other text.
"""

# Small talk system prompt
smalltalk_system_prompt = """
You are a friendly customer service representative for TechZone, a consumer electronics retailer.
You are handling casual conversation — greetings, thank-yous, farewells, and general chitchat.
Keep your response short, warm, and professional.
Always remind the customer that you can help with: warranty questions, tech support,
and deals or upgrade recommendations.
"""

# Instantiate the router agent
router_agent = RouterAgent(
    model          = model,
    system_prompt  = router_system_prompt,
    smalltalk_prompt = smalltalk_system_prompt,
    debug          = False   # Set to True to see routing decisions printed
)

print("TechZone RouterAgent instantiated successfully.")

TechZone RouterAgent instantiated successfully.


In [ ]:
# Test 1: Warranty Agent
config  = {"configurable": {"thread_id": str(uuid.uuid4())}}
messages = [HumanMessage(content="My laptop screen cracked after 8 months. Is that covered under warranty?")]
result   = router_agent.router_graph.invoke({"messages": messages}, config)
for message in result["messages"]:
    print(message.pretty_repr())

================================ Human Message =================================

My laptop screen cracked after 8 months. Is that covered under warranty?
================================== Ai Message ==================================

WARRANTY
================================== Ai Message ==================================

I'm sorry to hear about your laptop screen. Unfortunately, a cracked screen is considered physical damage, which is not covered under the standard warranty. However, if you purchased the extended TechZone+ warranty, which includes accidental damage protection, this might be covered. Could you please provide your order number and the device name so I can check your warranty status?


In [ ]:
# Test 2: Tech Support Agent
config  = {"configurable": {"thread_id": str(uuid.uuid4())}}
messages = [HumanMessage(content="My Bluetooth headphones keep disconnecting every few minutes. How do I fix this?")]
result   = router_agent.router_graph.invoke({"messages": messages}, config)
for message in result["messages"]:
    print(message.pretty_repr())

================================ Human Message =================================

My Bluetooth headphones keep disconnecting every few minutes. How do I fix this?
================================== Ai Message ==================================

TECHSUPPORT
================================== Ai Message ==================================

I'm sorry to hear your Bluetooth headphones keep disconnecting. Let's try to fix this together. 

First, could you please tell me:
- The brand and model of your headphones?
- The device you're connecting them to (phone, laptop, etc.) and its operating system version?
- Are you seeing any error messages when they disconnect?

In the meantime, here are some simple steps you can try:

1. **Restart both devices:** Turn off your headphones and the device you're connecting to, then turn them back on.
2. **Forget and re-pair:** On your device, go to Bluetooth settings, remove (forget) the headphones, then pair them again.
3. **Check for updates:** Make sure yo

In [ ]:
# Test 3: Deals & Upgrades Agent
config  = {"configurable": {"thread_id": str(uuid.uuid4())}}
messages = [HumanMessage(content="I want to upgrade my 3-year-old laptop for college. My budget is around $800.")]
result   = router_agent.router_graph.invoke({"messages": messages}, config)
for message in result["messages"]:
    print(message.pretty_repr())

================================ Human Message =================================

I want to upgrade my 3-year-old laptop for college. My budget is around $800.
================================== Ai Message ==================================

DEALS
================================== Ai Message ==================================

Great! Upgrading your laptop for college with a budget of $800 is totally doable, especially with our current promotions.

To help you best, could you tell me a bit about what you'll mainly use the laptop for? For example, will you be doing a lot of multitasking, graphic design, programming, or just general browsing and document work?

Meanwhile, here are a few options that fit your budget and needs:

1. **TechZone UltraBook 14"** – Lightweight and powerful with an Intel i5 processor, 8GB RAM, and 256GB SSD. Perfect for students who need portability and solid performance for multitasking and note-taking.  
   - With the 15% off laptops promo (code TECHWEEK), thi

In [ ]:
user_inputs = [
    "Hey there, my name is Rohan!",
    "My tablet keeps freezing and restarting on its own. It started two days ago.",
    "I bought it 6 months ago. Could this be a warranty issue?",
    "Actually, before I file a claim, are there any good deals on a replacement tablet?",
    "Thanks so much, that's everything I needed!"
]

# Single thread ID maintains memory across all 5 turns
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

for user_input in user_inputs:
    print(f"{'─'*50}")
    print(f"USER  : {user_input}")
    user_message = {"messages": [HumanMessage(user_input)]}
    ai_response  = router_agent.router_graph.invoke(user_message, config=config)
    print(f"\nAGENT : {ai_response['messages'][-1].content}")

──────────────────────────────────────────────────
USER  : Hey there, my name is Rohan!

AGENT : Hi Rohan! Great to meet you. If you ever need help with warranty questions, tech support, or finding the best deals and upgrades, just let me know! How can I assist you today?
──────────────────────────────────────────────────
USER  : My tablet keeps freezing and restarting on its own. It started two days ago.

AGENT : I'm sorry to hear your tablet is freezing and restarting on its own, Rohan. Let's work together to get this fixed.

To start, could you please tell me:
1. The brand and model of your tablet?
2. The operating system version it's running (for example, Android 11, iOS 15)?
3. Are you seeing any error messages or specific behavior before it restarts?

This info will help me guide you better!
──────────────────────────────────────────────────
USER  : I bought it 6 months ago. Could this be a warranty issue?

AGENT : Since your tablet is only 6 months old and is experiencing freezi

In [ ]:
memory_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

memory_inputs = [
    "Hi, my name is Rohan and I'm having issues with my TechZone Pro Tablet, model TZ-900.",
    "The screen flickers constantly. I've tried restarting it but the problem comes back.",
    "Do you remember what device I mentioned and what problem I described?"
]

for user_input in memory_inputs:
    print(f"{'─'*50}")
    print(f"USER  : {user_input}")
    user_message = {"messages": [HumanMessage(user_input)]}
    ai_response  = router_agent.router_graph.invoke(user_message, config=memory_config)
    print(f"\nAGENT : {ai_response['messages'][-1].content}")

──────────────────────────────────────────────────
USER  : Hi, my name is Rohan and I'm having issues with my TechZone Pro Tablet, model TZ-900.

AGENT : Hi Rohan! I’m sorry to hear you’re having issues with your TechZone Pro Tablet TZ-900. Could you please tell me more about the problem you’re experiencing? For example, is the tablet not turning on, running slowly, showing error messages, or something else? Any details you can provide will help me assist you better.
──────────────────────────────────────────────────
USER  : The screen flickers constantly. I've tried restarting it but the problem comes back.

AGENT : Thanks for the info, Rohan. Since the screen flickers even after restarting, let's try a few more steps:

1. **Check for Software Updates:**  
   - Go to **Settings** > **System** > **Software Update**.  
   - If an update is available, please download and install it, then see if the flickering stops.

2. **Adjust Display Settings:**  
   - Go to **Settings** > **Display**

In [ ]:
import builtins

chat_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("TechZone Customer Service Chatbot is ready!")
print("I can help with: warranty questions, tech support, and deals & upgrades.")
print("Type 'quit' to exit.\n")

while True:
    user_input = builtins.input("YOU   : ")

    if user_input.lower() in ["quit", "exit", "bye"]:
        print("AGENT : Thanks for contacting TechZone! Have a great day! 👋")
        break

    if user_input.strip() == "":
        continue

    user_message = {"messages": [HumanMessage(user_input)]}
    ai_response  = router_agent.router_graph.invoke(user_message, config=chat_config)
    print(f"\nAGENT : {ai_response['messages'][-1].content}\n")

TechZone Customer Service Chatbot is ready!
I can help with: warranty questions, tech support, and deals & upgrades.
Type 'quit' to exit.


AGENT : Hi Roy! I’m sorry to hear you’re having issues with your TechZone TZ-900 tablet. Could you please tell me more about the problem you’re experiencing? For example, is the tablet not turning on, running slowly, showing error messages, or something else? Any details you can provide will help me assist you better.


AGENT : Thanks for letting me know, Roy. Since the screen flickers and restarting didn’t help, let’s try a few more steps:

1. **Check for Software Updates:**  
   - Go to **Settings** > **System** > **Software Update** (or similar) and see if there’s an update available. Installing the latest update can fix bugs that cause screen issues.

2. **Adjust Screen Brightness and Refresh Rate:**  
   - Sometimes flickering can be related to brightness or refresh rate settings. Try lowering the brightness and see if the flickering changes. 